=====================================================
"""
STEP 1 AND 2. MODEL AND DATA PREPARATION, GENERATE CONTINUOUS TREATMENT

To test the inference framework, you need the prompt distribution $p(x)$, a reference policy $\pi_0$, and an optimal policy $\pi^*$.

The Dataset: Pull the TL;DR dataset (e.g., via Hugging Face datasets). Filter for a subset of "meaningful questions"—perhaps prompts (articles) that are sufficiently long or complex to generate variance in the quality of the summaries.

The Models: You have two choices here. You can either train your own models, or use off-the-shelf checkpoints to save compute. You will need a base reference model $\pi_0$ (like Mistral-7B-Instruct or Llama-3) and its exact DPO-fine-tuned counterpart $\pi^*$.
"""

This script handles the data ingestion, filtering, and continuous mapping. It loads an off-the-shelf reference model ($\pi_0$) to sample multiple candidate summaries (your continuous treatments) for complex prompts, and then utilizes a dense retriever to project these discrete texts into your continuous treatment space $\mathbb{R}^d$.

The architecture mirrors the robust PyTorch environment setups from your previous exp_llm_eval.py and Fenchel duality scripts, ensuring seamless integration with your existing workflows.

Model Pairs: The script defaults to HuggingFaceH4/zephyr-7b-sft-full as the reference policy ($\pi_0$). This is an excellent open-weights choice because it has an exact DPO-fine-tuned counterpart (HuggingFaceH4/zephyr-7b-dpo-full) that you can utilize in Step 2 for exact implicit reward extractions.

=====================================================

"""
DEBUGGING FOR STEP 1:
"""

The correct official reference model (the SFT baseline $\pi_0$ used to train Zephyr Beta) is actually HuggingFaceH4/mistral-7b-sft-beta, and its DPO counterpart ($\pi^*$) is HuggingFaceH4/zephyr-7b-beta.Since you already have the script saved, you don't need to rewrite the code. You can just override the broken default by passing the correct --pi_0_model argument in your terminal:

python step1_data_prep.py --pi_0_model HuggingFaceH4/mistral-7b-sft-beta

(Alternatively, if you strictly wanted the "full" handbook versions rather than the beta versions, the correct IDs for those are alignment-handbook/zephyr-7b-sft-full and alignment-handbook/zephyr-7b-dpo-full).

Run that updated command, and the script will successfully download the open weights and begin sampling your candidate summaries without needing an authentication token! Let me know if it starts processing smoothly.

=====================================================



STEP 3. Implementing the Nuisance Estimators

To keep our pipeline perfectly aligned, it is worth clarifying that the step1_data_prep.py script provided in the previous turn actually knocked out both Data Preparation and the generation/embedding of the continuous treatments. It sampled the candidate answers from the reference model and mapped them into the continuous space $D$ via embeddings.

The immediate next logical step in the pipeline is Implicit Reward Extraction. Because you are evaluating a Direct Preference Optimization (DPO) setup, you do not need to train a separate reward network (like the SimpleNet in your older exp_llm_eval.py). Instead, the optimal DPO policy $\pi^*$ mathematically encapsulates the reward.  We can compute the exact implicit reward for every candidate answer $y$ given prompt $x$ using the closed-form DPO relationship:

$$r^*(x,y) = \beta \log\left(\frac{\pi^*(y|x)}{\pi_0(y|x)}\right)$$

Reward Estimation: For each prompt-answer pair, compute the implicit reward using the DPO closed-form solution: $\hat{r}(x, y) = \beta \log(\pi^*(y|x)/\pi_0(y|x))$.

Green Density Estimation: Implement the regular Green density estimator $\hat{g}_0$. You will need to approximate the jump kernel $K(y_1, y_2|x) = \mu'(\hat{r}(x, y_1) - \hat{r}(x, y_2))$ and the exit rates $\hat{d}_x(y)$ by numerically integrating over the reference policy samples.

========================
STEP 3.5: GREEN DENSITY ESTIMATOR
This script implements the variational risk minimization to train the regular Green density estimator ($\hat{g}_0$). It bridges the gap between your theoretical integral operators and the empirical LLM setting by utilizing the pre-computed embeddings and exact implicit DPO rewards from the previous steps.  Because we need the prompt $X$ to serve as the contextual condition for the Green density $g_0(X, y_1, y_2)$, this script will automatically embed the prompt_text into the same $\mathbb{R}^d$ continuous space as your candidate summaries before initializing the training loop. 

====================
we map the theoretical foundations of Sections 4.5 and 4.6 into a complete execution script.

This script carries out Step 4 (Debiased Pointwise Estimation) and Step 5 (Hypothesis Testing) by:  Identifying the top candidate summary ($a_1$) and runner-up summary ($a_2$) for a test prompt based on raw implicit DPO rewards.  Solving the Empirical Riesz Representer problem (Equation 4.33) via a neural network to estimate the point-evaluation representer $\hat{\alpha}_{\Delta} = \hat{\alpha}_{s_1} - \hat{\alpha}_{s_2}$.  Constructing the plug-in score $\Psi_{\hat{\alpha}_{\Delta}}$ incorporating your pre-trained regular Green density network $\hat{g}_0$.  Calculating the sample variance $\hat{\sigma}_\Delta^2$ to execute the localized one-sided $Z$-test. 

NOTES:

Script Operational Flow

* Dynamic Riesz Resolution: The script avoids rigid structural choices by framing the pointwise evaluation as a standalone optimizer step for each tested prompt. It directly trains alpha_net to act as the evaluation direction $\alpha_{\Delta} = \alpha_{s_1} - \alpha_{s_2}$.
* Double Robust Correction: It constructs $\hat{g}_{\alpha}$ analytically across the batch samples, using the pre-trained weights from Step 3 to ensure that the leading first-order reward estimation errors are cleanly canceled.
* Statistical Decoupling: The final $p$-value provides a formal metric to satisfy Professor Lu's core constraint—uncovering whether the "top candidate" picked by a fine-tuned LLM retains a statistically significant edge after removing preference-data bias.